# SVM Classifier

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

from sklearn.svm import SVC

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv("dataset1.csv")
df.columns = df.columns.str.strip()
df = df.drop("index", axis=1, errors="ignore")

df.head()

,having_IPhaving_IP_Address,URLURL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


In [3]:
X = df.drop(columns=["Result"])
y = df["Result"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## Simple SVM Classifier

In [4]:
scaler_svm = StandardScaler()

X_train_scaled = scaler_svm.fit_transform(X_train)
X_test_scaled = scaler_svm.transform(X_test)

svm_model = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
error_rate_svm = 1 - accuracy_svm

print("Simple SVM Results")
print("------------------")
print("Accuracy:", accuracy_svm)
print("Error Rate:", error_rate_svm)
print()

print("Confusion Matrix:")
M_svm = confusion_matrix(y_test, y_pred_svm, labels=[-1, 1])

ConfusionMatrix_svm = pd.DataFrame(
    M_svm,
    index=["Legitimate (-1)", "Phishing (1)"],
    columns=["Legitimate (-1)", "Phishing (1)"]
)

ConfusionMatrix_svm["total"] = ConfusionMatrix_svm.sum(axis=1)

print(ConfusionMatrix_svm)
print()

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    labels=[-1, 1],
    target_names=["Legitimate (-1)", "Phishing (1)"]
))

Simple SVM Results
------------------
Accuracy: 0.9516056083220262
Error Rate: 0.04839439167797377

Confusion Matrix:
                 Legitimate (-1)  Phishing (1)  total
Legitimate (-1)              906            74    980
Phishing (1)                  33          1198   1231

Classification Report:
                 precision    recall  f1-score   support

Legitimate (-1)       0.96      0.92      0.94       980
   Phishing (1)       0.94      0.97      0.96      1231

       accuracy                           0.95      2211
      macro avg       0.95      0.95      0.95      2211
   weighted avg       0.95      0.95      0.95      2211



## Tuned SVM Model using Manual Cross-Validation

In [5]:
C_options = [0.1, 1, 10, 100]
gamma_options = ["scale", 0.01, 0.1, 1]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

svm_cv_results = []

for C_value in C_options:
    for gamma_value in gamma_options:
        
        fold_accuracies = []
        
        for train_index, val_index in cv.split(X_train, y_train):
            
            X_fold_train = X_train.iloc[train_index]
            X_fold_val = X_train.iloc[val_index]
            y_fold_train = y_train.iloc[train_index]
            y_fold_val = y_train.iloc[val_index]
            
            # Scale inside each fold to avoid data leakage
            scaler_fold = StandardScaler()
            X_fold_train_scaled = scaler_fold.fit_transform(X_fold_train)
            X_fold_val_scaled = scaler_fold.transform(X_fold_val)
            
            svm_fold = SVC(
                kernel="rbf",
                C=C_value,
                gamma=gamma_value,
                random_state=42
            )
            
            svm_fold.fit(X_fold_train_scaled, y_fold_train)
            
            y_fold_pred = svm_fold.predict(X_fold_val_scaled)
            fold_accuracy = accuracy_score(y_fold_val, y_fold_pred)
            
            fold_accuracies.append(fold_accuracy)
        
        svm_cv_results.append({
            "C": C_value,
            "gamma": gamma_value,
            "Mean CV Accuracy": np.mean(fold_accuracies),
            "CV Std": np.std(fold_accuracies)
        })

svm_cv_results_df = pd.DataFrame(svm_cv_results)

print("SVM Cross-Validation Results")
print("----------------------------")
print(svm_cv_results_df.sort_values(by="Mean CV Accuracy", ascending=False))

SVM Cross-Validation Results
----------------------------
        C  gamma  Mean CV Accuracy    CV Std
10   10.0    0.1          0.967097  0.001568
12  100.0  scale          0.965853  0.003088
14  100.0    0.1          0.964835  0.002561
8    10.0  scale          0.964044  0.003720
6     1.0    0.1          0.960765  0.003302
13  100.0   0.01          0.959182  0.005031
4     1.0  scale          0.949457  0.001659
9    10.0   0.01          0.948779  0.002490
5     1.0   0.01          0.934871  0.002936
0     0.1  scale          0.932383  0.003337
1     0.1   0.01          0.923903  0.003562
2     0.1    0.1          0.891452  0.008042
11   10.0      1          0.866012  0.006759
15  100.0      1          0.866012  0.006759
7     1.0      1          0.863412  0.007359
3     0.1      1          0.563998  0.001609


In [6]:
best_svm_row = svm_cv_results_df.sort_values(
    by="Mean CV Accuracy",
    ascending=False
).iloc[0]

best_C_svm = best_svm_row["C"]
best_gamma_svm = best_svm_row["gamma"]

print("Best C:", best_C_svm)
print("Best gamma:", best_gamma_svm)
print("Best Mean CV Accuracy:", best_svm_row["Mean CV Accuracy"])

Best C: 10.0
Best gamma: 0.1
Best Mean CV Accuracy: 0.9670966673402412


In [7]:
scaler_svm_tuned = StandardScaler()

X_train_scaled_tuned = scaler_svm_tuned.fit_transform(X_train)
X_test_scaled_tuned = scaler_svm_tuned.transform(X_test)

best_svm = SVC(
    kernel="rbf",
    C=best_C_svm,
    gamma=best_gamma_svm,
    random_state=42
)

best_svm.fit(X_train_scaled_tuned, y_train)

y_pred_svm_tuned = best_svm.predict(X_test_scaled_tuned)

accuracy_svm_tuned = accuracy_score(y_test, y_pred_svm_tuned)
error_rate_svm_tuned = 1 - accuracy_svm_tuned

print("Tuned SVM Results")
print("-----------------")
print("Accuracy:", accuracy_svm_tuned)
print("Error Rate:", error_rate_svm_tuned)
print()

print("Confusion Matrix:")
M_svm_tuned = confusion_matrix(y_test, y_pred_svm_tuned, labels=[-1, 1])

ConfusionMatrix_svm_tuned = pd.DataFrame(
    M_svm_tuned,
    index=["Legitimate (-1)", "Phishing (1)"],
    columns=["Legitimate (-1)", "Phishing (1)"]
)

ConfusionMatrix_svm_tuned["total"] = ConfusionMatrix_svm_tuned.sum(axis=1)

print(ConfusionMatrix_svm_tuned)
print()

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred_svm_tuned,
    labels=[-1, 1],
    target_names=["Legitimate (-1)", "Phishing (1)"]
))

Tuned SVM Results
-----------------
Accuracy: 0.973767526006332
Error Rate: 0.026232473993668015

Confusion Matrix:
                 Legitimate (-1)  Phishing (1)  total
Legitimate (-1)              938            42    980
Phishing (1)                  16          1215   1231

Classification Report:
                 precision    recall  f1-score   support

Legitimate (-1)       0.98      0.96      0.97       980
   Phishing (1)       0.97      0.99      0.98      1231

       accuracy                           0.97      2211
      macro avg       0.97      0.97      0.97      2211
   weighted avg       0.97      0.97      0.97      2211



## SVM with Feature Engineering

In [9]:
poly_svm = PolynomialFeatures(
    degree=2,
    interaction_only=False,
    include_bias=False
)

X_train_interactions = poly_svm.fit_transform(X_train)
X_test_interactions = poly_svm.transform(X_test)

interaction_feature_names = poly_svm.get_feature_names_out(X.columns)

print("Original number of features:", X_train.shape[1])
print("Number of features after adding interaction terms:", X_train_interactions.shape[1])

Original number of features: 30
Number of features after adding interaction terms: 495


## Simple SVM Classifier with Feature Engineering

In [10]:
scaler_svm_interactions = StandardScaler()

X_train_interactions_scaled = scaler_svm_interactions.fit_transform(X_train_interactions)
X_test_interactions_scaled = scaler_svm_interactions.transform(X_test_interactions)

svm_interactions_model = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    random_state=42
)

svm_interactions_model.fit(X_train_interactions_scaled, y_train)

y_pred_svm_interactions = svm_interactions_model.predict(X_test_interactions_scaled)

accuracy_svm_interactions = accuracy_score(y_test, y_pred_svm_interactions)
error_rate_svm_interactions = 1 - accuracy_svm_interactions

print("Simple SVM with Interaction Features Results")
print("--------------------------------------------")
print("Accuracy:", accuracy_svm_interactions)
print("Error Rate:", error_rate_svm_interactions)
print()

print("Confusion Matrix:")
M_svm_interactions = confusion_matrix(y_test, y_pred_svm_interactions, labels=[-1, 1])

ConfusionMatrix_svm_interactions = pd.DataFrame(
    M_svm_interactions,
    index=["Legitimate (-1)", "Phishing (1)"],
    columns=["Legitimate (-1)", "Phishing (1)"]
)

ConfusionMatrix_svm_interactions["total"] = ConfusionMatrix_svm_interactions.sum(axis=1)

print(ConfusionMatrix_svm_interactions)
print()

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred_svm_interactions,
    labels=[-1, 1],
    target_names=["Legitimate (-1)", "Phishing (1)"]
))

Simple SVM with Interaction Features Results
--------------------------------------------
Accuracy: 0.9615558570782451
Error Rate: 0.03844414292175491

Confusion Matrix:
                 Legitimate (-1)  Phishing (1)  total
Legitimate (-1)              927            53    980
Phishing (1)                  32          1199   1231

Classification Report:
                 precision    recall  f1-score   support

Legitimate (-1)       0.97      0.95      0.96       980
   Phishing (1)       0.96      0.97      0.97      1231

       accuracy                           0.96      2211
      macro avg       0.96      0.96      0.96      2211
   weighted avg       0.96      0.96      0.96      2211



## Tuned SVM with Feature Engineering using Manual Cross-Validation

In [11]:
C_options_interactions = [0.1, 1, 10]
gamma_options_interactions = ["scale", 0.01, 0.1]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

svm_interactions_cv_results = []

for C_value in C_options_interactions:
    for gamma_value in gamma_options_interactions:
        
        fold_accuracies = []
        
        for train_index, val_index in cv.split(X_train_interactions, y_train):
            
            X_fold_train = X_train_interactions[train_index]
            X_fold_val = X_train_interactions[val_index]
            y_fold_train = y_train.iloc[train_index]
            y_fold_val = y_train.iloc[val_index]
            
            # Scale inside each fold to avoid data leakage
            scaler_fold = StandardScaler()
            X_fold_train_scaled = scaler_fold.fit_transform(X_fold_train)
            X_fold_val_scaled = scaler_fold.transform(X_fold_val)
            
            svm_fold = SVC(
                kernel="rbf",
                C=C_value,
                gamma=gamma_value,
                random_state=42
            )
            
            svm_fold.fit(X_fold_train_scaled, y_fold_train)
            
            y_fold_pred = svm_fold.predict(X_fold_val_scaled)
            fold_accuracy = accuracy_score(y_fold_val, y_fold_pred)
            
            fold_accuracies.append(fold_accuracy)
        
        svm_interactions_cv_results.append({
            "C": C_value,
            "gamma": gamma_value,
            "Mean CV Accuracy": np.mean(fold_accuracies),
            "CV Std": np.std(fold_accuracies)
        })

svm_interactions_cv_results_df = pd.DataFrame(svm_interactions_cv_results)

print("SVM with Interaction Features Cross-Validation Results")
print("------------------------------------------------------")
print(svm_interactions_cv_results_df.sort_values(by="Mean CV Accuracy", ascending=False))

SVM with Interaction Features Cross-Validation Results
------------------------------------------------------
      C  gamma  Mean CV Accuracy    CV Std
6  10.0  scale          0.967436  0.002041
3   1.0  scale          0.958277  0.003631
7  10.0   0.01          0.948101  0.004185
4   1.0   0.01          0.946518  0.005303
0   0.1  scale          0.933401  0.004290
8  10.0    0.1          0.786978  0.017987
5   1.0    0.1          0.778271  0.016061
1   0.1   0.01          0.751582  0.009950
2   0.1    0.1          0.558005  0.000954


In [12]:
best_svm_interactions_row = svm_interactions_cv_results_df.sort_values(
    by="Mean CV Accuracy",
    ascending=False
).iloc[0]

best_C_svm_interactions = best_svm_interactions_row["C"]
best_gamma_svm_interactions = best_svm_interactions_row["gamma"]

print("Best C:", best_C_svm_interactions)
print("Best gamma:", best_gamma_svm_interactions)
print("Best Mean CV Accuracy:", best_svm_interactions_row["Mean CV Accuracy"])

Best C: 10.0
Best gamma: scale
Best Mean CV Accuracy: 0.9674355862273597


In [13]:
scaler_svm_interactions_tuned = StandardScaler()

X_train_interactions_scaled_tuned = scaler_svm_interactions_tuned.fit_transform(X_train_interactions)
X_test_interactions_scaled_tuned = scaler_svm_interactions_tuned.transform(X_test_interactions)

best_svm_interactions = SVC(
    kernel="rbf",
    C=best_C_svm_interactions,
    gamma=best_gamma_svm_interactions,
    random_state=42
)

best_svm_interactions.fit(X_train_interactions_scaled_tuned, y_train)

y_pred_svm_interactions_tuned = best_svm_interactions.predict(X_test_interactions_scaled_tuned)

accuracy_svm_interactions_tuned = accuracy_score(y_test, y_pred_svm_interactions_tuned)
error_rate_svm_interactions_tuned = 1 - accuracy_svm_interactions_tuned

print("Tuned SVM with Interaction Features Results")
print("-------------------------------------------")
print("Accuracy:", accuracy_svm_interactions_tuned)
print("Error Rate:", error_rate_svm_interactions_tuned)
print()

print("Confusion Matrix:")
M_svm_interactions_tuned = confusion_matrix(y_test, y_pred_svm_interactions_tuned, labels=[-1, 1])

ConfusionMatrix_svm_interactions_tuned = pd.DataFrame(
    M_svm_interactions_tuned,
    index=["Legitimate (-1)", "Phishing (1)"],
    columns=["Legitimate (-1)", "Phishing (1)"]
)

ConfusionMatrix_svm_interactions_tuned["total"] = ConfusionMatrix_svm_interactions_tuned.sum(axis=1)

print(ConfusionMatrix_svm_interactions_tuned)
print()

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred_svm_interactions_tuned,
    labels=[-1, 1],
    target_names=["Legitimate (-1)", "Phishing (1)"]
))

Tuned SVM with Interaction Features Results
-------------------------------------------
Accuracy: 0.973767526006332
Error Rate: 0.026232473993668015

Confusion Matrix:
                 Legitimate (-1)  Phishing (1)  total
Legitimate (-1)              940            40    980
Phishing (1)                  18          1213   1231

Classification Report:
                 precision    recall  f1-score   support

Legitimate (-1)       0.98      0.96      0.97       980
   Phishing (1)       0.97      0.99      0.98      1231

       accuracy                           0.97      2211
      macro avg       0.97      0.97      0.97      2211
   weighted avg       0.97      0.97      0.97      2211

